# Context Compression (Enterprise AI Pattern)

Context Compression is one of the **most important optimization techniques** in Advanced RAG.

It solves one of the biggest production problems:

> **The retriever finds many relevant documents, but the LLM has a limited context window and processing unnecessary text increases cost, latency, and hallucinations.**

Interviewers frequently ask:

- What is Context Compression?
- Why do we need it?
- Is it the same as summarization?
- How does it improve RAG?
- Explain the architecture.

---

# 1. What is Context Compression?

## Definition

**Context Compression** is the process of reducing retrieved documents to only the information relevant to the user's query before sending them to the LLM.

Instead of sending the entire retrieved document, only the relevant paragraphs or sentences are passed to the model.

---

## Interview Answer

> Context Compression is an Advanced RAG technique that filters or compresses retrieved documents so that only the information relevant to the user's question is sent to the LLM. This reduces token usage, latency, cost, and hallucinations while improving response quality.

---

# 2. Why Do We Need Context Compression?

Suppose the retriever returns:

```text id="ctx1"
HR Policy

↓

20 Pages
```

User asks

```text id="ctx2"
How many annual leave days are provided?
```

Without Context Compression

```text id="ctx3"
LLM receives

20 Pages
```

Most of those pages are irrelevant.

Problems

- Higher token cost
- Higher latency
- More hallucinations
- Context window may overflow

---

With Context Compression

```text id="ctx4"
LLM receives

Employees receive 20 annual leave days.
Unused leave can be carried forward for 5 days.
```

Only the relevant section.

---

# 3. Traditional RAG

```text id="ctx5"
Retriever

↓

Top 5 Documents

↓

LLM
```

The LLM reads everything.

---

# 4. Context Compression

```text id="ctx6"
Retriever

↓

Top 5 Documents

↓

Context Compressor

↓

Relevant Sentences

↓

LLM
```

Only useful information reaches the LLM.

---

# 5. Architecture

```text id="ctx7"
                    User Question
                           │
                           ▼
                     Hybrid Search
                           │
                           ▼
                    Retrieved Chunks
                           │
                           ▼
                 Context Compressor
                           │
         ┌─────────────────┴─────────────────┐
         ▼                                   ▼
 Remove Irrelevant Text              Keep Relevant Text
         └─────────────────┬─────────────────┘
                           ▼
                  Compressed Context
                           ▼
             AWS Bedrock / Azure OpenAI
                           ▼
                      Final Answer
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Retriever | OpenSearch/Qdrant | Azure AI Search/Qdrant |
| Embeddings | Titan | text-embedding-3-large |
| Backend | ECS/EKS | Container Apps/AKS |

---

# 6. Example

Document

```text id="ctx8"
HR Policy

Annual Leave

Employees receive 20 annual leave days.

Unused leave can be carried forward.

Medical Leave

Employees receive 15 medical leave days.

Travel Policy

Employees receive travel allowance.
```

---

Question

```text id="ctx9"
How many annual leave days are provided?
```

---

Compressed Context

```text id="ctx10"
Employees receive 20 annual leave days.
```

Instead of sending the whole document.

---

# 7. Enterprise Flow

```text id="ctx11"
Question

↓

Retriever

↓

Top 10 Chunks

↓

Context Compression

↓

Best Paragraphs

↓

Bedrock

↓

Answer
```

---

# 8. How Context Compression Works

There are several approaches.

## A. LLM-Based Compression

The LLM reads retrieved chunks and removes irrelevant information.

---

## B. Embedding-Based Compression

Each sentence is embedded.

Only the most similar sentences are retained.

---

## C. Reranker-Based Compression

A reranker scores passages.

Only the highest-scoring passages are forwarded.

---

# 9. LangChain Example

```python
# ==========================================================
# STEP 1 : Create Base Retriever
#
# Purpose:
# Retrieve relevant documents from
# Qdrant/OpenSearch.
# ==========================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)


# ==========================================================
# STEP 2 : Create Bedrock LLM
#
# Purpose:
# Used for contextual compression.
# ==========================================================

from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1"
)


# ==========================================================
# STEP 3 : Create LLM Compressor
#
# Purpose:
# Remove irrelevant text from retrieved documents.
# ==========================================================

from langchain.retrievers.document_compressors import (
    LLMChainExtractor
)

compressor = LLMChainExtractor.from_llm(llm)


# ==========================================================
# STEP 4 : Create Contextual Compression Retriever
#
# Purpose:
# Wrap the normal retriever with a
# compression layer.
# ==========================================================

from langchain.retrievers import (
    ContextualCompressionRetriever
)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)


# ==========================================================
# STEP 5 : Retrieve Compressed Documents
# ==========================================================

docs = compression_retriever.invoke(
    "How many annual leave days are provided?"
)

for doc in docs:
    print(doc.page_content)
```

---

# 10. Production Architecture

```text id="ctx12"
User

↓

FastAPI

↓

JWT

↓

LangGraph

↓

Query Rewriting

↓

Hybrid Search

↓

Reranker

↓

Context Compression

↓

Bedrock

↓

Redis

↓

Response
```

---

# 11. Advantages

✅ Lower token usage

✅ Lower cost

✅ Faster responses

✅ Better context quality

✅ Reduced hallucinations

---

# 12. Disadvantages

❌ Additional processing step

❌ Slight increase in retrieval latency

❌ Over-compression may remove useful context

❌ Requires tuning

---

# 13. Best Practices

✅ Compress **after retrieval**, not before.

✅ Combine with Hybrid Search and reranking.

✅ Preserve important metadata.

✅ Keep enough surrounding context for the LLM.

---

# 14. Common Mistakes

❌ Compressing too aggressively.

❌ Removing supporting paragraphs.

❌ Losing metadata during compression.

❌ Using compression on already short documents.

---

# 15. Context Compression vs Summarization

| Context Compression | Summarization |
|---------------------|---------------|
| Removes irrelevant content | Creates a shorter summary |
| Keeps original wording | Rewrites content |
| Used before LLM generation | Often used as a final output |
| Retrieval optimization | User-facing content generation |

Example

Original

```text id="ctx13"
20-page HR policy
```

Compression

```text id="ctx14"
Annual leave section only
```

Summary

```text id="ctx15"
Employees receive paid annual leave according to company policy.
```

---

# 16. Context Compression vs Reranking

| Reranking | Context Compression |
|------------|---------------------|
| Chooses the best documents | Chooses the best text within documents |
| Works at document level | Works at passage/sentence level |
| Before compression | After retrieval/reranking |

In production they are commonly used together.

---

# 17. Real Enterprise Example

## HR Assistant

Question

```text id="ctx16"
How many annual leave days are provided?
```

Retriever returns

- Leave Policy
- Medical Policy
- Travel Policy
- Payroll Policy

Compression keeps only

```text id="ctx17"
Employees receive 20 annual leave days.
```

---

## Healthcare Assistant

Question

```text id="ctx18"
Contraindications of Metformin
```

Retriever returns

- Diabetes guideline
- Nutrition guide
- Exercise guide

Compression keeps only

```text id="ctx19"
Contraindications section from the diabetes guideline.
```

---

# 18. Enterprise Pipeline

```text id="ctx20"
PDF

↓

Semantic Chunking

↓

Parent-Child Retrieval

↓

Query Rewriting

↓

Hybrid Search

↓

Multi-Query Retrieval

↓

Self-Query Retriever

↓

Reranker

↓

Context Compression

↓

Bedrock / Azure OpenAI

↓

Response
```

This is a common production pipeline for high-quality enterprise RAG systems.

---

# 19. Common Interview Questions

### Q1. Why Context Compression?

To reduce token usage and send only the information relevant to the user's question.

---

### Q2. Does Context Compression reduce hallucinations?

Yes.

By removing irrelevant context, the LLM has fewer distractions and produces more focused answers.

---

### Q3. Is Context Compression mandatory?

No.

It becomes valuable when retrieved documents are long or numerous.

---

### Q4. Can Context Compression be combined with Hybrid Search?

Yes.

A common production pipeline is:

```text id="ctx21"
Hybrid Search

↓

Reranker

↓

Context Compression

↓

LLM
```

---

### Q5. Where is Context Compression useful?

- HR policy assistants
- Healthcare guidelines
- Legal AI
- Financial reports
- Technical documentation
- Large enterprise knowledge bases

---

# 20. Complete Advanced RAG Pipeline

```text id="ctx22"
User Question
       │
       ▼
Query Rewriting
       │
       ▼
Multi-Query Retrieval
       │
       ▼
Hybrid Search (BM25 + Vector)
       │
       ▼
Self-Query Retriever (Metadata Filters)
       │
       ▼
Parent-Child Retrieval
       │
       ▼
Reranker
       │
       ▼
Context Compression
       │
       ▼
AWS Bedrock / Azure OpenAI
       │
       ▼
Final Response
```

---

# 21. EPAM Senior Answer (3 Minutes)

> "Context Compression is an Advanced RAG optimization technique that reduces the amount of retrieved text sent to the LLM by keeping only the information relevant to the user's question. After retrieval—typically using Hybrid Search, metadata filtering, or Parent-Child Retrieval—a compression step removes irrelevant paragraphs or sentences while preserving the key context required to answer the query. This significantly reduces token consumption, lowers inference cost, improves latency, and helps reduce hallucinations caused by unrelated context. In LangChain, this can be implemented using a `ContextualCompressionRetriever` together with an LLM-based extractor such as `LLMChainExtractor`. In production, I typically place Context Compression after reranking and before Amazon Bedrock or Azure OpenAI so that only the highest-quality, most relevant context reaches the model. This approach is particularly effective for long enterprise documents such as HR policies, healthcare guidelines, legal contracts, and technical manuals."